In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
llm.invoke("안녕")

'하세요, 저는 새로운 팀원이에요\n\n제 이름은 조남경이에요\n\n저는 컴퓨터 공학을 전공했고, 개발자로 일하고 있어요.\n\n새로운 팀에 합류해서 함께 열심히 일하고 성장할 수 있기를 기대합니다.\n\n저는 새로운 것을 배우는 것을 좋아하고, 도전하는 것을 두려워하지 않아요.\n\n또한 새로운 사람들과 어울리는 것을 즐기고, 함께 문제를 해결하는 것을 좋아해요.\n\n앞으로 잘 부탁드립니다. 감사합니다.'

In [7]:
# <오픈AI API를 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
import openai
from typing import List

# 기본 오픈AI 클라이언트 사용
client = openai.OpenAI()

# "안녀하세요!" 메시지를 보내고 응답을 받음
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕하세요!"}]
)
response.choices[0].message.content

'안녕하세요! 어떻게 도와드릴까요?'

In [8]:
# 요청에 사용할 프롬프트 템플릿 정의
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요"

# 메시지를 보내고 모델의 응답을 받는 함수
def call_chat_model(messages: List[dict]) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content

# 주어진 주제에 따라 설명을 요청하는 함수
def invoke_chain(topic: str) -> str:
    prompt_value = prompt_template.format(topic=topic)
    messages = [{"role": "user", "content": prompt_value}]
    return call_chat_model(messages)

# "더블딥" 주제로 설명 요청
invoke_chain("더블딥")

'더블딥(double dip)은 경제학 및 금융에서 사용하는 용어로, 경기 침체가 두 차례 이상 발생하는 상황을 말합니다. 일반적으로 첫 번째 경기 침체 후 경기가 반등을 하다 다시 하락하게 되는 경우를 설명합니다. 이 현상은 소비자 신뢰도 저하, 기업 투자 감소, 시장 불확실성 등으로 인해 발생할 수 있으며, 이러한 두 번의 침체를 모두 겪는 것이 더 큰 경기 둔화를 초래할 수 있습니다. 더블딥은 경제 정책 결정자들에게 큰 도전 과제가 되기도 합니다.'

In [9]:
# <랭체인을 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 미스트랄AI 모델을 사용할 경우 주석 해제
from langchain_mistralai.chat_models import ChatMistralAI

# 주어진 주제에 대해 짧은 설명을 요청하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "주제 {topic}에 대해 짧은 설명을 해주세요"
)

# 출력 파서를 문자열로 설정
output_parser = StrOutputParser()
# 오픈AI의 gpt-4o 모델을 사용하여 채팅 모델 설정
model = ChatOpenAI(model="gpt-4o")
# 미스트랄AI 모델을 사용할 경우 주석 해제
# model = ChatMistralAI()

# 파이프라인 설정: 주제를 받아 프롬프트를 생성하고, 모델을 통해 응답을 생성한 후 문자열로 파싱
chain = (
    {"topic": RunnablePassthrough()}  # 입력 받은 주제를 그대로 통과시킴
    | prompt                         # 프롬프트 템플릿을 적용
    | model                          # 모델을 사용해 응답 생성
    | output_parser                  # 응답을 문자열로 파싱
)

# "더블딥" 주제로 설명 요청
chain.invoke("위례신사선")

'위례신사선은 대한민국 서울특별시에서 계획 중인 도시철도 노선 중 하나입니다. 이 노선은 위례신도시와 서울 강남 지역을 연결하기 위해 설계되었습니다. 위례신사선은 시민들의 교통 편의를 높이고 수도권 남부 지역의 교통 혼잡을 완화시키는 것을 목표로 하고 있습니다. 총 연장 약 14.8km에 달하며, 여러 주요 거점을 연결할 예정입니다. 이 노선이 개통되면 위례신도시 주민들의 강남 접근성이 크게 향상될 것으로 기대됩니다. 프로젝트는 여러 단계에 걸쳐 진행 중이며, 구체적인 개통 일정은 변경될 수 있습니다.'